# Reveal: no_c7_security_cot — code-based v2

Dùng cache CodeT5/Joern đã xác minh của Reproduced-Code-Based. Chọn kernel WSL, Restart Kernel rồi Run All. Output mới: No-C7-security-cot/test_v3/. Cache cũ trong test/ được giữ. Reveal cần baseline prepare/freeze hoàn tất trước.

C6 dùng PDG có sẵn, có thay đổi nguồn biểu diễn graph so với cache legacy; manifest ghi rõ. C8 chỉ có tác dụng đa dạng hóa khi C2 bật nhiều demo. Xem README.md trong thư mục Ablations.


In [ ]:
from pathlib import Path
import os, sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents, Path('/mnt/c/MY-FILES/LAB-THAY-THO'), Path('C:/MY-FILES/LAB-THAY-THO')) if (p / 'Graduation-Project/Ablation_V2').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Cannot locate workspace root')
ABLATIONS = ROOT / 'Graduation-Project/Ablation_V2/Reveal/Ablations'
sys.path.insert(0, str(ABLATIONS))
from code_based_runner import AblationConfig, ARM_SPECS, run_stage
import code_based_runner as _lock_runner
sys.path.insert(0, str(ROOT / 'Graduation-Project/Ablation_V2'))
from notebook_resource_wait import configure_resource_wait
run_stage = configure_resource_wait(_lock_runner)
ARM = 'no_c7_security_cot'
RUN_INFERENCE = True
WORKERS, INFERENCE_WORKERS = 4, 10
config = AblationConfig(root=ROOT, dataset='reveal', workers=WORKERS, inference_workers=INFERENCE_WORKERS, run_inference=RUN_INFERENCE)
print('Arm:', ARM, '| enabled:', ARM_SPECS[ARM]['enabled'])
print('Output:', ABLATIONS / ARM_SPECS[ARM]['folder'] / 'test_v3')


## Prepare và freeze
Đọc cache nặng tại chỗ; không encode lại và không gọi Joern. Kiểm tra fingerprint trước khi tạo prompt.


In [ ]:
prepared = run_stage(config, ARM, 'prepare')
prepared


## Inference và evaluation
Cell này gọi Vertex thật nếu RUN_INFERENCE=True. Checkpoint mới nằm trong test_v3.


In [ ]:
if RUN_INFERENCE:
    metrics = run_stage(config, ARM, 'inference')
    result = run_stage(config, ARM, 'evaluate')
    print(result['status'], result['evaluation'])
else:
    print('Preparation/freeze complete; RUN_INFERENCE=False')
